In [1]:
# ----------------------------- logging --------------------------
import logging
from sys import stdout
from datetime import datetime
import os

logging.basicConfig(
    level=logging.INFO,
    format=f"[%(asctime)s][%(levelname)s][{os.environ.get('USERNAME')}] %(message)s",
    stream=stdout,
    datefmt="%m-%d %H:%M:%S",
)
logging.info(datetime.now())

import numpy as np


# ####################################################################
def gauss_jordan(A: np.ndarray) -> np.ndarray:
    """Resuelve un sistema de ecuaciones lineales mediante el método de Gauss-Jordan.

    ## Parameters

    ``A``: matriz aumentada del sistema de ecuaciones lineales. Debe ser de tamaño n-by-(n+1), donde n es el número de incógnitas.

    ## Return

    ``solucion``: vector con la solución del sistema de ecuaciones lineales.

    """
    if not isinstance(A, np.ndarray):
        logging.debug("Convirtiendo A a numpy array.")
        A = np.array(
            A, dtype=float
        )  # convertir en float, porque si no, convierte en enteros
    assert A.shape[0] == A.shape[1] - 1, "La matriz A debe ser de tamaño n-by-(n+1)."
    n = A.shape[0]

    for i in range(0, n):  # loop por columna

        # --- encontrar pivote
        p = None  # default, first element
        for pi in range(i, n):
            if A[pi, i] == 0:
                # must be nonzero
                continue

            if p is None:
                # first nonzero element
                p = pi
                continue

            if abs(A[pi, i]) < abs(A[p, i]):
                p = pi

        if p is None:
            # no pivot found.
            logging.info(f"\n{A}")
            raise ValueError("No existe solución única.")

        if p != i:
            logging.info(f"Intercambiando filas {i} y {p}.")
            # swap rows
            _aux = A[i, :].copy()
            A[i, :] = A[p, :].copy()
            A[p, :] = _aux

        # --- Eliminación: loop por fila
        for j in range(n):
            if i == j:
                continue
            m = A[j, i] / A[i, i]
            A[j, i:] = A[j, i:] - m * A[i, i:]

        logging.info(f"\n{A}")

    if A[n - 1, n - 1] == 0:
        # Sin embargo, esto solo se accede al finalizar la matriz... Con todos los pivotes
        if A[n - 1, n] == 0:
            raise ValueError("Infinitas soluciones.")
        else:
            raise ValueError("Sin solución.")

    # --- Sustitución hacia atrás
    solucion = np.zeros(n)
    for i in range(n):
        solucion[i] = A[i, n] / A[i, i]

    return solucion

[07-21 16:55:47][INFO][pinto] 2026-07-21 16:55:47.467164


In [2]:
A = [
    [1, 2, 3, 4, 1],
    [2, 5, 6, 7, -2],
    [3, 6, 8, 9, 3],
    [4, 7, 9, 10, 4],
]

gauss_jordan(A)

[07-21 16:55:49][INFO][pinto] 
[[ 1.  2.  3.  4.  1.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0. -1. -3. -6.  0.]]
[07-21 16:55:49][INFO][pinto] 
[[ 1.  0.  3.  6.  9.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0. -3. -7. -4.]]
[07-21 16:55:49][INFO][pinto] 
[[ 1.  0.  0. -3.  9.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0.  0.  2. -4.]]
[07-21 16:55:49][INFO][pinto] 
[[ 1.  0.  0.  0.  3.]
 [ 0.  1.  0.  0. -6.]
 [ 0.  0. -1.  0. -6.]
 [ 0.  0.  0.  2. -4.]]


array([ 3., -6.,  6., -2.])

# Resolver

In [3]:
import numpy as np


# ####################################################################
def inv_matrix(A: np.ndarray) -> np.ndarray:
    """Inversión de una matriz cuadrada mediante método de Gauss-Jordan.
    ## Parameters
    ``A``: matriz cuadrada de tamaño n x n.

    ## Return
    ``A_inv``: matriz inversa de A.
    """
    # Asegurar que A es un array de numpy y obtener su tamaño
    A = np.array(A, dtype=float)
    n = A.shape[0]
    
    # Validar que sea una matriz cuadrada
    assert A.shape[0] == A.shape[1], "La matriz A debe ser cuadrada."
    
    # Preparar la matriz identidad I y una matriz vacía para la inversa
    I = np.eye(n)
    A_inv = np.zeros((n, n))
    
    # Resolver A * x = e_i para cada columna de la matriz identidad
    for i in range(n):
        # Extraer la columna i de la identidad como un vector columna
        e_i = I[:, i].reshape(-1, 1)
        
        # Crear la matriz aumentada [A | e_i]
        # NOTA: Usamos A.copy() porque gauss_jordan modifica la matriz in-place
        A_aug = np.hstack((A.copy(), e_i))
        
        # Obtener la solución usando la función proporcionada
        solucion_columna = gauss_jordan(A_aug)
        
        # Asignar la solución a la columna i de la matriz inversa
        A_inv[:, i] = solucion_columna
        
    return A_inv

## Ejemplos
* Ejemplo 1

In [4]:
# La matriz A =
A = [
    [1, 2, 3, 4],
    [2, 5, 6, 7],
    [3, 6, 8, 9],
    [4, 7, 9, 10],
]
# tiene como inversa
# A_inv =[[ 0.5, -0.5, -1.5,  1.5],
#        [-0.5,  1.5, -1.5,  0.5],
#        [-1.5, -1.5,  3.5, -1.5],
#        [ 1.5,  0.5, -1.5,  0.5]]
inv_matrix(A)

[07-21 16:55:55][INFO][pinto] 
[[ 1.  2.  3.  4.  1.]
 [ 0.  1.  0. -1. -2.]
 [ 0.  0. -1. -3. -3.]
 [ 0. -1. -3. -6. -4.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.  0.  3.  6.  5.]
 [ 0.  1.  0. -1. -2.]
 [ 0.  0. -1. -3. -3.]
 [ 0.  0. -3. -7. -6.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.  0.  0. -3. -4.]
 [ 0.  1.  0. -1. -2.]
 [ 0.  0. -1. -3. -3.]
 [ 0.  0.  0.  2.  3.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.   0.   0.   0.   0.5]
 [ 0.   1.   0.   0.  -0.5]
 [ 0.   0.  -1.   0.   1.5]
 [ 0.   0.   0.   2.   3. ]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.  2.  3.  4.  0.]
 [ 0.  1.  0. -1.  1.]
 [ 0.  0. -1. -3.  0.]
 [ 0. -1. -3. -6.  0.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.  0.  3.  6. -2.]
 [ 0.  1.  0. -1.  1.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0. -3. -7.  1.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.  0.  0. -3. -2.]
 [ 0.  1.  0. -1.  1.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0.  0.  2.  1.]]
[07-21 16:55:55][INFO][pinto] 
[[ 1.   0.   0.   0.  -0.5]
 [ 0.   1.   0.   0.   1.5]
 [ 0.   0.  -1.   0.   1.

array([[ 0.5, -0.5, -1.5,  1.5],
       [-0.5,  1.5, -1.5,  0.5],
       [-1.5, -1.5,  3.5, -1.5],
       [ 1.5,  0.5, -1.5,  0.5]])

* Ejemplo 2

In [5]:
# La matriz A =
A = [
    [4, 4, 5, 1],
    [3, 4, 2, 2],
    [2, 1, 4, 1],
    [3, 2, 5, 4],
]
# tiene como inversa
# A_inv =[[-34.,  31.,  52., -20.],
#         [ 19., -17., -29.,  11.],
#         [ 12., -11., -18.,   7.],
#         [  1.,  -1.,  -2.,   1.]]
inv_matrix(A)

[07-21 16:55:59][INFO][pinto] Intercambiando filas 0 y 2.
[07-21 16:55:59][INFO][pinto] 
[[ 2.   1.   4.   1.   0. ]
 [ 0.   2.5 -4.   0.5  0. ]
 [ 0.   2.  -3.  -1.   1. ]
 [ 0.   0.5 -1.   2.5  0. ]]
[07-21 16:55:59][INFO][pinto] Intercambiando filas 1 y 3.
[07-21 16:55:59][INFO][pinto] 
[[  2.    0.    6.   -4.    0. ]
 [  0.    0.5  -1.    2.5   0. ]
 [  0.    0.    1.  -11.    1. ]
 [  0.    0.    1.  -12.    0. ]]
[07-21 16:55:59][INFO][pinto] 
[[  2.    0.    0.   62.   -6. ]
 [  0.    0.5   0.   -8.5   1. ]
 [  0.    0.    1.  -11.    1. ]
 [  0.    0.    0.   -1.   -1. ]]
[07-21 16:55:59][INFO][pinto] 
[[  2.    0.    0.    0.  -68. ]
 [  0.    0.5   0.    0.    9.5]
 [  0.    0.    1.    0.   12. ]
 [  0.    0.    0.   -1.   -1. ]]
[07-21 16:55:59][INFO][pinto] Intercambiando filas 0 y 2.
[07-21 16:55:59][INFO][pinto] 
[[ 2.   1.   4.   1.   0. ]
 [ 0.   2.5 -4.   0.5  1. ]
 [ 0.   2.  -3.  -1.   0. ]
 [ 0.   0.5 -1.   2.5  0. ]]
[07-21 16:55:59][INFO][pinto] Intercambiando f

array([[-34.,  31.,  52., -20.],
       [ 19., -17., -29.,  11.],
       [ 12., -11., -18.,   7.],
       [  1.,  -1.,  -2.,   1.]])

## Ejercicios

* Ejercicio 1

In [6]:
A = [[2, -3], [-1, 1]]
inv_matrix(A)

[07-21 16:56:02][INFO][pinto] Intercambiando filas 0 y 1.
[07-21 16:56:02][INFO][pinto] 
[[-1.  1.  0.]
 [ 0. -1.  1.]]
[07-21 16:56:02][INFO][pinto] 
[[-1.  0.  1.]
 [ 0. -1.  1.]]
[07-21 16:56:02][INFO][pinto] Intercambiando filas 0 y 1.
[07-21 16:56:02][INFO][pinto] 
[[-1.  1.  1.]
 [ 0. -1.  2.]]
[07-21 16:56:02][INFO][pinto] 
[[-1.  0.  3.]
 [ 0. -1.  2.]]


array([[-1., -3.],
       [-1., -2.]])

* Ejercicio 2

In [7]:
A = [
    [4, 0, 0, 5],
    [1, 0, 4, 0],
    [3, 4, 1, 3],
    [1, 3, 3, 0],
]
inv_matrix(A)

[07-21 16:56:05][INFO][pinto] Intercambiando filas 0 y 1.
[07-21 16:56:05][INFO][pinto] 
[[  1.   0.   4.   0.   0.]
 [  0.   0. -16.   5.   1.]
 [  0.   4. -11.   3.   0.]
 [  0.   3.  -1.   0.   0.]]
[07-21 16:56:05][INFO][pinto] Intercambiando filas 1 y 3.
[07-21 16:56:05][INFO][pinto] 
[[  1.           0.           4.           0.           0.        ]
 [  0.           3.          -1.           0.           0.        ]
 [  0.           0.          -9.66666667   3.           0.        ]
 [  0.           0.         -16.           5.           1.        ]]
[07-21 16:56:05][INFO][pinto] 
[[ 1.          0.          0.          1.24137931  0.        ]
 [ 0.          3.          0.         -0.31034483  0.        ]
 [ 0.          0.         -9.66666667  3.          0.        ]
 [ 0.          0.          0.          0.03448276  1.        ]]
[07-21 16:56:05][INFO][pinto] 
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -3.60000000e+01]
 [ 0.00000000e+00  3.00000000e+00  0

array([[-36.,  45.,  60., -80.],
       [  3.,  -4.,  -5.,   7.],
       [  9., -11., -15.,  20.],
       [ 29., -36., -48.,  64.]])

* Ejercicio 3

In [8]:
A = [
    [0, 0, 0, 0, 0, 0, 1, -1],
    [0, 1, -1, 1, 0, -1, 0, 1],
    [-1, -1, 0, 0, 2, 1, 0, 0],
    [-1, -1, -1, 1, 2, 0, 0, 1],
    [-1, 1, 1, 0, -1, -1, 0, 2],
    [0, 1, 0, 0, -1, -1, 0, 0],
    [1, -1, -1, 1, 2, 1, 0, 2],
    [2, 0, 0, 0, 0, 1, 2, 0],
]
inv_matrix(A)

[07-21 16:56:08][INFO][pinto] Intercambiando filas 0 y 2.
[07-21 16:56:08][INFO][pinto] 
[[-1. -1.  0.  0.  2.  1.  0.  0.  0.]
 [ 0.  1. -1.  1.  0. -1.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  1.]
 [ 0.  0. -1.  1.  0. -1.  0.  1.  0.]
 [ 0.  2.  1.  0. -3. -2.  0.  2.  0.]
 [ 0.  1.  0.  0. -1. -1.  0.  0.  0.]
 [ 0. -2. -1.  1.  4.  2.  0.  2.  0.]
 [ 0. -2.  0.  0.  4.  3.  2.  0.  0.]]
[07-21 16:56:08][INFO][pinto] 
[[-1.  0. -1.  1.  2.  0.  0.  1.  0.]
 [ 0.  1. -1.  1.  0. -1.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  1.]
 [ 0.  0. -1.  1.  0. -1.  0.  1.  0.]
 [ 0.  0.  3. -2. -3.  0.  0.  0.  0.]
 [ 0.  0.  1. -1. -1.  0.  0. -1.  0.]
 [ 0.  0. -3.  3.  4.  0.  0.  4.  0.]
 [ 0.  0. -2.  2.  4.  1.  2.  2.  0.]]
[07-21 16:56:08][INFO][pinto] Intercambiando filas 2 y 3.
[07-21 16:56:08][INFO][pinto] 
[[-1.  0.  0.  0.  2.  1.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0. -1.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  1.]
 [ 0.  0.

array([[ 2., -1., -0., -1., -0.,  2.,  2., -1.],
       [ 0.,  1.,  1., -1.,  0.,  0.,  0.,  0.],
       [ 6., -1., -0., -3.,  1.,  1.,  4., -3.],
       [ 6.,  1., -1., -3.,  1., -3.,  3., -3.],
       [ 2., -1.,  1., -1., -0.,  3.,  2., -1.],
       [-2.,  2., -0., -0., -0., -4., -2.,  1.],
       [-1.,  0.,  0.,  1.,  0.,  0., -1.,  1.],
       [-2.,  0.,  0.,  1.,  0.,  0., -1.,  1.]])

* Ejercicio 4

In [9]:
A = [
    [1, 0, 0, 0, -1, 0, 0, -1, 1, -1],
    [1, 1, 0, -1, -1, 1, 0, 0, 1, -1],
    [-1, 0, -1, 0, 0, 0, -1, 1, 0, 0],
    [0, 0, -1, 0, -1, -1, 1, 0, 1, 0],
    [-1, 0, 0, -1, 1, 1, 1, 1, 0, -1],
    [1, 0, 0, 1, -1, -1, -1, 1, -1, 0],
    [1, 1, 1, 0, 1, 0, -1, -1, -1, 1],
    [1, 1, 1, 1, 0, 0, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 0, -1, -1, 0, 0],
    [0, 0, -1, -1, -1, 0, 1, 1, 1, -1],
]
inv_matrix(A)

[07-21 16:56:11][INFO][pinto] 
[[ 1.  0.  0.  0. -1.  0.  0. -1.  1. -1.  1.]
 [ 0.  1.  0. -1.  0.  1.  0.  1.  0.  0. -1.]
 [ 0.  0. -1.  0. -1.  0. -1.  0.  1. -1.  1.]
 [ 0.  0. -1.  0. -1. -1.  1.  0.  1.  0.  0.]
 [ 0.  0.  0. -1.  0.  1.  1.  0.  1. -2.  1.]
 [ 0.  0.  0.  1.  0. -1. -1.  2. -2.  1. -1.]
 [ 0.  1.  1.  0.  2.  0. -1.  0. -2.  2. -1.]
 [ 0.  1.  1.  1.  1.  0.  1.  2. -1.  1. -1.]
 [ 0.  1.  1.  1.  2.  0. -1.  0. -1.  1. -1.]
 [ 0.  0. -1. -1. -1.  0.  1.  1.  1. -1.  0.]]
[07-21 16:56:11][INFO][pinto] 
[[ 1.  0.  0.  0. -1.  0.  0. -1.  1. -1.  1.]
 [ 0.  1.  0. -1.  0.  1.  0.  1.  0.  0. -1.]
 [ 0.  0. -1.  0. -1.  0. -1.  0.  1. -1.  1.]
 [ 0.  0. -1.  0. -1. -1.  1.  0.  1.  0.  0.]
 [ 0.  0.  0. -1.  0.  1.  1.  0.  1. -2.  1.]
 [ 0.  0.  0.  1.  0. -1. -1.  2. -2.  1. -1.]
 [ 0.  0.  1.  1.  2. -1. -1. -1. -2.  2.  0.]
 [ 0.  0.  1.  2.  1. -1.  1.  1. -1.  1.  0.]
 [ 0.  0.  1.  2.  2. -1. -1. -1. -1.  1.  0.]
 [ 0.  0. -1. -1. -1.  0.  1.  1.  1. -1.  0

array([[ 14.,  -8.,   9.,  -4.,   0.,  -4.,   9.,   7.,  -8.,   3.],
       [ -2.,   2.,  -1.,   2.,   1.,   1.,  -1.,  -1.,   1.,  -2.],
       [-27.,  14., -18.,   5.,  -2.,   7., -17., -13.,  16.,  -2.],
       [ 12.,  -6.,   8.,  -2.,   1.,  -3.,   7.,   6.,  -7.,  -0.],
       [  6.,  -4.,   4.,  -2.,   0.,  -2.,   4.,   3.,  -3.,   2.],
       [ 18.,  -9.,  12.,  -4.,   1.,  -5.,  11.,   9., -11.,   1.],
       [  8.,  -4.,   5.,  -1.,   1.,  -2.,   5.,   4.,  -5.,  -0.],
       [ -5.,   2.,  -3.,   0.,  -1.,   1.,  -3.,  -2.,   3.,   1.],
       [-11.,   5.,  -7.,   1.,  -2.,   2.,  -7.,  -5.,   7.,   1.],
       [  1.,  -1.,   1.,  -1.,  -1.,  -1.,   1.,   1.,  -1.,   1.]])